# Knowledge Distillation: Compressing Large Vision-Language Models (VLMs)
**Group No:** 105  
**Course:** Conversational AI  

## Objective
Implement an end-to-end Knowledge Distillation pipeline where a massive 'Teacher' model (CLIP) transfers its learned continuous embedding space to a lightweight 'Student' model (DistilBERT + MobileNetV2). The goal is to optimize the model for CPU-bound edge deployment in semantic image-text retrieval tasks, achieving a massive reduction in size and latency while preserving semantic recall.

---

## Task A: Library imports and Dataset Preparation (Flickr30k)
We utilize the **Flickr30k** dataset via Hugging Face. To simulate production data processing and ensure the vision encoders receive standard inputs, we wrap the dataset in a PyTorch `Dataset` class to apply ImageNet normalization `(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])` and resize all images to `224x224`.

*Note: For efficient execution within the Virtual Lab compute constraints, a subset of 1,000 samples is used.*

In [1]:
# Sample dataset import snippet for reference (taken from HuggingFace)
# from datasets import load_dataset

# ds = load_dataset("lmms-lab-encoder/flickr30k")

In [2]:
import os
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
from transformers import CLIPModel, CLIPProcessor, DistilBertModel, DistilBertConfig, AutoTokenizer

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing assignment on device: {DEVICE}")

d:\06_HigherStudies\01_AI_ML\Subjects\03_Semester03\Conversational AI\Assignments\Assignment 2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Executing assignment on device: cuda


In [3]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch

class HFFlickrWrapper(Dataset):
    """
    Wraps a Hugging Face dataset to apply PyTorch image transformations 
    and format the outputs for our training loop.
    """
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        
        # Hugging Face datasets return PIL images directly
        # We ensure it's RGB to prevent crashes on grayscale images
        image = item['image'].convert("RGB")
        
        # Extract caption. Note: Some HF datasets provide a list of captions per image.
        # If 'caption' is a list, we take the first one: item['caption'][0]
        # Adjust 'caption' key if the specific HF repo uses a different name (e.g., 'text')
        caption = item['caption'] 
        if isinstance(caption, list):
            caption = caption[0]
            
        if self.transform:
            image = self.transform(image)
            
        return {"image": image, "caption": caption, "idx": idx}

In [4]:
'''
# # 1. Load the dataset directly from Hugging Face
# print("Downloading/Loading Flickr30k from Hugging Face...")
# # You can use the specific repo you mentioned, or standard ones
# raw_dataset = load_dataset("lmms-lab-encoder/flickr30k", split="test", cache_dir="./dataset/flickr30k_data") # Using 'test' split or standard 'train'

# # Optional: Subset the data for faster lab execution (e.g., 5,000 samples)
# raw_dataset = raw_dataset.select(range(5000))

# # 2. Split into Train/Test using Hugging Face's built-in method
# split_ds = raw_dataset.train_test_split(test_size=0.2, seed=42)

# # 3. Wrap with PyTorch Dataset for transformations
# train_ds = HFFlickrWrapper(split_ds['train'])
# test_ds = HFFlickrWrapper(split_ds['test'])

# # 4. Create PyTorch DataLoaders
# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
# test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
'''

'\n# # 1. Load the dataset directly from Hugging Face\n# print("Downloading/Loading Flickr30k from Hugging Face...")\n# # You can use the specific repo you mentioned, or standard ones\n# raw_dataset = load_dataset("lmms-lab-encoder/flickr30k", split="test", cache_dir="./dataset/flickr30k_data") # Using \'test\' split or standard \'train\'\n\n# # Optional: Subset the data for faster lab execution (e.g., 5,000 samples)\n# raw_dataset = raw_dataset.select(range(5000))\n\n# # 2. Split into Train/Test using Hugging Face\'s built-in method\n# split_ds = raw_dataset.train_test_split(test_size=0.2, seed=42)\n\n# # 3. Wrap with PyTorch Dataset for transformations\n# train_ds = HFFlickrWrapper(split_ds[\'train\'])\n# test_ds = HFFlickrWrapper(split_ds[\'test\'])\n\n# # 4. Create PyTorch DataLoaders\n# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)\n# test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)\n'

In [5]:
# 1. Load Dataset via Hugging Face
print("Downloading/Loading Flickr30k from Hugging Face...")
# We use a try-except to fallback to a standard dataset if 'lmms-lab-encoder' is unavailable
try:
    raw_dataset = load_dataset("lmms-lab-encoder/flickr30k", split="test", cache_dir="./dataset/flickr30k_data") # Using 'test' split or standard 'train'

except:
    raw_dataset = load_dataset("nlphq/flickr30k", split="test", cache_dir="/dataset/flickr30k_data")

# Subset to 1000 samples to ensure the Virtual Lab executes efficiently
subset_dataset = raw_dataset.select(range(1000))
split_ds = subset_dataset.train_test_split(test_size=0.2, seed=42)

train_ds = HFFlickrWrapper(split_ds['train'])
test_ds = HFFlickrWrapper(split_ds['test'])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)


Downloading/Loading Flickr30k from Hugging Face...


# Module 1: Teacher Embedding Generation & Student Setup

## Task 1: Teacher Model Integration & Soft-Target Generation
The teacher is **OpenAI's CLIP-ViT-Base-Patch32**. 
Unlike discrete distillation (which maps categorical probability logits), we are performing **continuous distillation**. We freeze the teacher's weights entirely so it acts as a stable anchor, generating high-dimensional ($512$-d) geometric "soft targets" for the student to mimic. The teacher uses its own internal BPE tokenizer to prevent vocabulary misalignment.

### Teacher Integration

In [6]:
class TeacherVLM(nn.Module):
    """
    Task 1: Teacher Model with self-contained CLIP Tokenizer/Processor
    """
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        super().__init__()
        print(f"Loading Teacher Model: {model_name}...")
        self.clip = CLIPModel.from_pretrained(model_name)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        
        for param in self.clip.parameters():
            param.requires_grad = False

    def forward(self, raw_captions, pixel_values):
        device = pixel_values.device
        with torch.no_grad():
            # Use CLIP's OWN processor for text encoding
            inputs = self.processor(text=raw_captions, return_tensors="pt", padding=True, truncation=True).to(device)
            text_outputs = self.clip.text_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            text_embeds = self.clip.text_projection(text_outputs.pooler_output)
            text_embeds = F.normalize(text_embeds, p=2, dim=-1)

            # Vision encoding
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
            image_embeds = self.clip.visual_projection(vision_outputs.pooler_output)
            image_embeds = F.normalize(image_embeds, p=2, dim=-1)

            return image_embeds, text_embeds

In [7]:
# 2. Initialize Models & Optimizers
teacher = TeacherVLM().to(DEVICE)

Loading Teacher Model: openai/clip-vit-base-patch32...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 34583.24it/s]


## Task 2: Student Tokenizer & Vision Projection Alignment
To optimize for edge environments, the student uses a shallow **2-layer DistilBERT** (Text) and **MobileNetV2** (Vision). 

Because the student encoders output mismatched dimensions ($256$-d and $1280$-d, respectively), we append **linear projection heads** to map both modalities explicitly into the teacher's $512$-d continuous space.

In [8]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Module 2: Distillation Architecture & Training
The objective function balances self-supervised contrastive learning with teacher-guided feature alignment:

$$ \mathcal{L}_{\text{total}} = \alpha \mathcal{L}_{\text{distill}} + (1 - \alpha) \mathcal{L}_{\text{contrastive}} $$

We utilize **Cosine Distance** to map the student into the teacher's manifold, combined with symmetric **InfoNCE** cross-entropy for image-text pair matching.

We train two student models for a comparative analysis:
1. **Student WITH Distillation:** Guided by the CLIP Teacher ($\alpha=0.8$).
2. **Student WITHOUT Distillation (Baseline):** Supervised entirely by contrastive loss ($\alpha=0.0$).

## Task 3: Compact Student Vision-Language Transformer Construction

### Student model

In [9]:
class StudentVLM(nn.Module):
    """
    Task 2 & 3: Compact Dual-Encoder Architecture.
    """
    def __init__(self, teacher_embed_dim=512):
        super().__init__()
        text_config = DistilBertConfig(n_layers=2, dim=256, hidden_dim=512, n_heads=4)
        self.text_encoder = DistilBertModel(text_config)
        self.text_proj = nn.Linear(256, teacher_embed_dim)
        
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.vision_encoder = mobilenet.features
        self.vision_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.vision_proj = nn.Linear(1280, teacher_embed_dim)
        
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, input_ids, attention_mask, pixel_values):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = text_out.last_hidden_state[:, 0, :] 
        text_emb = self.text_proj(text_cls)
        
        vision_feat = self.vision_encoder(pixel_values)
        vision_feat = self.vision_pool(vision_feat).flatten(1)
        image_emb = self.vision_proj(vision_feat)
        
        image_emb = F.normalize(image_emb, p=2, dim=-1)
        text_emb = F.normalize(text_emb, p=2, dim=-1)
        
        return image_emb, text_emb

## Task 4: Feature Distillation Loss Function
The custom loss function combines:
1. **Cosine Distance Loss ($\mathcal{L}_{\text{distill}}$):** Forces student embeddings to match the geometric orientation of the teacher's representations.
2. **InfoNCE Contrastive Loss ($\mathcal{L}_{\text{contrastive}}$):** Symmetric cross-entropy over batch image-text pairs.

$$\mathcal{L}_{\text{total}} = \alpha \mathcal{L}_{\text{distill}} + (1 - \alpha) \mathcal{L}_{\text{contrastive}}$$

In [10]:
class DistillationLoss(nn.Module):
    """
    Task 4: Custom Distillation Loss using Cosine Distance + InfoNCE
    """
    def __init__(self, alpha=0.7, scale=10.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, s_img_emb, s_txt_emb, t_img_emb, t_txt_emb, logit_scale):
        # 1. Feature Alignment via Cosine Distance (1 - Cosine Similarity)
        loss_distill_img = (1.0 - F.cosine_similarity(s_img_emb, t_img_emb)).mean()
        loss_distill_txt = (1.0 - F.cosine_similarity(s_txt_emb, t_txt_emb)).mean()
        loss_distill = (loss_distill_img + loss_distill_txt) / 2.0

        # 2. InfoNCE Contrastive Loss
        scale = torch.clamp(logit_scale.exp(), max=100.0)
        logits_img2txt = scale * s_img_emb @ s_txt_emb.t()
        logits_txt2img = logits_img2txt.t()

        batch_size = s_img_emb.size(0)
        targets = torch.arange(batch_size, device=s_img_emb.device)

        loss_c_img = F.cross_entropy(logits_img2txt, targets)
        loss_c_txt = F.cross_entropy(logits_txt2img, targets)
        loss_contrastive = (loss_c_img + loss_c_txt) / 2.0

        # Weighted combination (based on alpha)
        total_loss = (self.alpha * loss_distill) + ((1.0 - self.alpha) * loss_contrastive)
        return total_loss, loss_distill, loss_contrastive

In [11]:
# epochs
epochs = 15

#Note: training will be carried on after definition of contrastive_criterion function
student_baseline = StudentVLM().to(DEVICE)
student_distilled = StudentVLM().to(DEVICE)

distill_criterion = DistillationLoss(alpha=0.4)
contrastive_criterion = DistillationLoss(alpha=0.0)

# Change lr from 1e-4 to 5e-4 for faster projection convergence
opt_distilled = torch.optim.AdamW(student_distilled.parameters(), lr=1e-4)
opt_baseline = torch.optim.AdamW(student_baseline.parameters(), lr=1e-4)

d:\06_HigherStudies\01_AI_ML\Subjects\03_Semester03\Conversational AI\Assignments\Assignment 2\.venv\lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\06_HigherStudies\01_AI_ML\Subjects\03_Semester03\Conversational AI\Assignments\Assignment 2\.venv\lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [12]:
# 3. Train Distilled Student
print("\n--- Training Distilled Student ---")
student_distilled.train()
for epoch in range(epochs):
    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        tokens = tokenizer(batch["caption"], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        
        opt_distilled.zero_grad()
        t_img_emb, t_txt_emb = teacher(batch["caption"], images)
        s_img_emb, s_txt_emb = student_distilled(tokens["input_ids"], tokens["attention_mask"], images)
        
        loss, _, _ = distill_criterion(s_img_emb, s_txt_emb, t_img_emb, t_txt_emb, student_distilled.logit_scale)
        loss.backward()
        opt_distilled.step()
    print(f"Epoch {epoch+1} Complete.")


--- Training Distilled Student ---
Epoch 1 Complete.
Epoch 2 Complete.
Epoch 3 Complete.
Epoch 4 Complete.
Epoch 5 Complete.
Epoch 6 Complete.
Epoch 7 Complete.
Epoch 8 Complete.
Epoch 9 Complete.
Epoch 10 Complete.
Epoch 11 Complete.
Epoch 12 Complete.
Epoch 13 Complete.
Epoch 14 Complete.
Epoch 15 Complete.


In [13]:
# 4. Train Baseline Student
print("\n--- Training Baseline Student ---")
student_baseline.train()
for epoch in range(epochs):
    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        tokens = tokenizer(batch["caption"], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        
        opt_baseline.zero_grad()
        s_img_emb, s_txt_emb = student_baseline(tokens["input_ids"], tokens["attention_mask"], images)
        
        loss, _, _ = contrastive_criterion(s_img_emb, s_txt_emb, s_img_emb.detach(), s_txt_emb.detach(), student_baseline.logit_scale)
        loss.backward()
        opt_baseline.step()
    print(f"Epoch {epoch+1} Complete.")


--- Training Baseline Student ---
Epoch 1 Complete.
Epoch 2 Complete.
Epoch 3 Complete.
Epoch 4 Complete.
Epoch 5 Complete.
Epoch 6 Complete.
Epoch 7 Complete.
Epoch 8 Complete.
Epoch 9 Complete.
Epoch 10 Complete.
Epoch 11 Complete.
Epoch 12 Complete.
Epoch 13 Complete.
Epoch 14 Complete.
Epoch 15 Complete.


In [14]:
import os

# Create local folder for models
save_dir = "./saved_models"
os.makedirs(save_dir, exist_ok=True)

# Define paths
distilled_path = os.path.join(save_dir, "distilled_student.pth")
baseline_path = os.path.join(save_dir, "baseline_student.pth")

# Save state dicts
torch.save(student_distilled.state_dict(), distilled_path)
torch.save(student_baseline.state_dict(), baseline_path)

print(f"Distilled model successfully saved to: {distilled_path}")
print(f"Baseline model successfully saved to:  {baseline_path}")

Distilled model successfully saved to: ./saved_models\distilled_student.pth
Baseline model successfully saved to:  ./saved_models\baseline_student.pth



# Module 3: Comparative Analysis & Efficiency Benchmarking
We define functions to evaluate the semantic retrieval accuracy (**Recall@1, 5, 10**) and to profile the physical deployment metrics of the models (**Model Size in MB and Inference Latency in ms/query**).

## Task 5: Retrieval Performance Evaluation
We evaluate image-to-text retrieval across three models:
1. The Teacher VLM (CLIP-Base)
2. Student trained WITHOUT Knowledge Distillation (Baseline)
3. Student trained WITH Knowledge Distillation

### Function for evaluating retrieval performance

In [15]:
def evaluate_retrieval(model, dataloader, student_tokenizer, is_teacher=False):
    model.eval()
    all_img_embeds, all_txt_embeds = [], []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(DEVICE)
            captions = batch["caption"]
            
            if is_teacher:
                # Pass raw captions so Teacher uses its own CLIP tokenizer
                img_emb, txt_emb = model(captions, images)
            else:
                # Student uses DistilBERT tokenizer
                tokens = student_tokenizer(captions, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
                img_emb, txt_emb = model(tokens["input_ids"], tokens["attention_mask"], images)

            all_img_embeds.append(img_emb.cpu())
            all_txt_embeds.append(txt_emb.cpu())

    img_embeds = torch.cat(all_img_embeds, dim=0)
    txt_embeds = torch.cat(all_txt_embeds, dim=0)
    sim_matrix = img_embeds @ txt_embeds.t()
    
    ranks = []
    for i in range(sim_matrix.size(0)):
        sorted_indices = torch.argsort(sim_matrix[i], descending=True)
        rank = (sorted_indices == i).nonzero(as_tuple=True)[0].item()
        ranks.append(rank)

    ranks = np.array(ranks)
    return np.mean(ranks < 1)*100, np.mean(ranks < 5)*100, np.mean(ranks < 10)*100

### Evaluate and benchmark the three models

In [16]:
# 5. Evaluate and Benchmark
print("\nEvaluating Models...")
t_r1, t_r5, t_r10 = evaluate_retrieval(teacher, test_loader, tokenizer, is_teacher=True)
b_r1, b_r5, b_r10 = evaluate_retrieval(student_baseline, test_loader, tokenizer, is_teacher=False)
d_r1, d_r5, d_r10 = evaluate_retrieval(student_distilled, test_loader, tokenizer, is_teacher=False)


Evaluating Models...


### Print final tables

In [17]:
# 6. Print Final Tables
print("\n" + "="*60 + "\nTASK 5: RETRIEVAL EVALUATION RESULTS\n" + "="*60)
print(f"{'Model':<35} | {'R@1 (%)':<8} | {'R@5 (%)':<8} | {'R@10 (%)':<8}")
print("-" * 68)
print(f"{'1. Teacher (CLIP-Base)':<35} | {t_r1:<8.1f} | {t_r5:<8.1f} | {t_r10:<8.1f}")
print(f"{'2. Student (w/o Distillation)':<35} | {b_r1:<8.1f} | {b_r5:<8.1f} | {b_r10:<8.1f}")
print(f"{'3. Student (WITH Distillation)':<35} | {d_r1:<8.1f} | {d_r5:<8.1f} | {d_r10:<8.1f}")


TASK 5: RETRIEVAL EVALUATION RESULTS
Model                               | R@1 (%)  | R@5 (%)  | R@10 (%)
--------------------------------------------------------------------
1. Teacher (CLIP-Base)              | 85.5     | 96.0     | 99.5    
2. Student (w/o Distillation)       | 6.0      | 18.0     | 27.0    
3. Student (WITH Distillation)      | 4.5      | 13.0     | 22.5    


## Task 6: Deployment Metrics Analysis
Calculating storage size (MB) and average single-query inference latency (ms/query) on CPU.

### Placeholder function for benchmarking model size and inference latency

In [18]:
def benchmark_deployment_metrics(model, sample_batch, tokenizer, is_teacher=False):
    """
    Calculates model size (MB) and average CPU inference latency per query.
    """
    model.eval()
    
    # 1. Calculate Model Size (MB)
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / (1024 * 1024)

    # 2. Benchmark CPU Latency
    cpu_device = torch.device("cpu")
    model_cpu = model.to(cpu_device)
    images = sample_batch["image"].to(cpu_device)
    captions = sample_batch["caption"]

    # Warmup runs
    with torch.no_grad():
        for _ in range(5): 
            if is_teacher:
                _ = model_cpu(captions, images)
            else:
                tokens = tokenizer(captions, padding=True, truncation=True, return_tensors="pt").to(cpu_device)
                _ = model_cpu(tokens["input_ids"], tokens["attention_mask"], images)

    # Benchmark runs
    start_time = time.time()
    num_runs = 20
    with torch.no_grad():
        for _ in range(num_runs):
            if is_teacher:
                _ = model_cpu(captions, images)
            else:
                tokens = tokenizer(captions, padding=True, truncation=True, return_tensors="pt").to(cpu_device)
                _ = model_cpu(tokens["input_ids"], tokens["attention_mask"], images)
                
    avg_latency_ms = ((time.time() - start_time) / (num_runs * len(images))) * 1000

    return model_size_mb, avg_latency_ms

### Test inference latency

In [19]:
sample_batch = next(iter(test_loader))

# Calculate metrics
s_size, s_latency = benchmark_deployment_metrics(student_distilled, sample_batch, tokenizer, is_teacher=False)
t_size, t_latency = benchmark_deployment_metrics(teacher, sample_batch, tokenizer, is_teacher=True)



print("\n" + "="*60 + "\nTASK 6: DEPLOYMENT METRICS BENCHMARK\n" + "="*60)
print(f"{'Metric':<25} | {'Teacher (CLIP)':<15} | {'Distilled Student':<15}")
print("-" * 60)
print(f"{'Model Size (MB)':<25} | {t_size:<15.2f} | {s_size:<15.2f}")
print(f"{'CPU Latency (ms/query)':<25} | {t_latency:<15.2f} | {s_latency:<15.2f}")
print("="*60)


TASK 6: DEPLOYMENT METRICS BENCHMARK
Metric                    | Teacher (CLIP)  | Distilled Student
------------------------------------------------------------
Model Size (MB)           | 577.08          | 45.95          
CPU Latency (ms/query)    | 51.55           | 9.05           
